# 🏥 CodiEsp NER Training — Google Colab

Trains the Spanish Biomedical BERT NER model on CodiEsp using a free Colab T4 GPU.

## Before You Start
1. **Runtime → Change runtime type → T4 GPU** (or better)
2. Zip your project locally (run in your terminal — takes ~5 seconds):
   ```bash
   cd ~/Documents/my/career/demos
   zip -r medical-coding.zip medical-coding/ \
     --exclude '*/data/*' \
     --exclude '*/models/*' \
     --exclude '*/__pycache__/*' \
     --exclude '*/.git/*' \
     --exclude '*.pyc'
   ```
3. Upload `medical-coding.zip` to your **Google Drive root** (or any folder — update `ZIP_PATH` below)
4. Run all cells top to bottom (**Runtime → Run all**)

**Estimated training time:**
| GPU | Full 5 epochs |
|---|---|
| T4 (free Colab) | ~25–40 min |
| A100 (Colab Pro) | ~8–12 min |

## 0. Check GPU

In [ ]:
import torch

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU: {gpu}  ({vram:.1f} GB VRAM)")
else:
    print("❌ No GPU detected — go to Runtime → Change runtime type → T4 GPU")
    raise SystemExit("Stopping: GPU required")

## 1. Mount Google Drive & Unzip Project

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ── Update this path if you uploaded the zip to a subfolder ──
ZIP_PATH = '/content/drive/MyDrive/medical-coding.zip'
WORK_DIR = '/content/medical-coding'

import os, zipfile, shutil

if os.path.exists(WORK_DIR):
    print(f"Project already exists at {WORK_DIR}, skipping unzip.")
else:
    print(f"Unzipping {ZIP_PATH} → {WORK_DIR} …")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall('/content/')
    print("Done.")

os.chdir(WORK_DIR)
print(f"Working directory: {os.getcwd()}")
print("Contents:", os.listdir('.'))

## 2. Install Dependencies

In [ ]:
# Colab already has torch/transformers — only install what's missing
!pip install -q \
    transformers==4.46.* \
    datasets \
    sentence-transformers \
    faiss-cpu \
    seqeval \
    tqdm \
    accelerate

print("\n✅ Dependencies installed")

## 3. Download CodiEsp Dataset

In [ ]:
import sys
sys.path.insert(0, '.')

DATA_DIR = 'data'
DRIVE_DATA_CACHE = '/content/drive/MyDrive/medical-coding-data'  # persist data to Drive

os.makedirs(DATA_DIR, exist_ok=True)

# Check if data already exists in Drive cache
if os.path.exists(f'{DRIVE_DATA_CACHE}/train/trainX.tsv'):
    print("Data found in Drive cache — symlinking to avoid re-downloading…")
    for split in ['train', 'dev', 'test', 'background']:
        src = f'{DRIVE_DATA_CACHE}/{split}'
        dst = f'{DATA_DIR}/{split}'
        if os.path.exists(src) and not os.path.exists(dst):
            os.symlink(src, dst)
    print("✅ Data linked from Drive cache")
else:
    print("Downloading CodiEsp from Zenodo (~11 MB)…")
    !python scripts/download_data.py

    # Save to Drive for future runs
    print(f"\nCopying data to Drive cache at {DRIVE_DATA_CACHE} (so you don't re-download next time)…")
    os.makedirs(DRIVE_DATA_CACHE, exist_ok=True)
    for split in ['train', 'dev', 'test']:
        src = f'{DATA_DIR}/{split}'
        dst = f'{DRIVE_DATA_CACHE}/{split}'
        if not os.path.exists(dst):
            shutil.copytree(src, dst)
    print("✅ Data cached to Drive")

# Quick sanity check
from scripts.download_data import print_structure
from pathlib import Path
print_structure(Path(DATA_DIR))

## 4. Build ICD-10 Index

In [ ]:
DRIVE_ICD10_CM  = '/content/drive/MyDrive/medical-coding-data/icd10cm_descriptions.json'
DRIVE_ICD10_PCS = '/content/drive/MyDrive/medical-coding-data/icd10pcs_descriptions.json'

# Restore from Drive if already built
if os.path.exists(DRIVE_ICD10_CM):
    import shutil
    shutil.copy(DRIVE_ICD10_CM,  'data/icd10cm_descriptions.json')
    shutil.copy(DRIVE_ICD10_PCS, 'data/icd10pcs_descriptions.json')
    import json
    cm_n  = len(json.load(open('data/icd10cm_descriptions.json')))
    pcs_n = len(json.load(open('data/icd10pcs_descriptions.json')))
    print(f"✅ ICD-10 index restored from Drive — CM: {cm_n:,}  PCS: {pcs_n:,}")
else:
    !python scripts/build_icd10_index.py
    # Save to Drive
    import shutil
    shutil.copy('data/icd10cm_descriptions.json',  DRIVE_ICD10_CM)
    shutil.copy('data/icd10pcs_descriptions.json', DRIVE_ICD10_PCS)
    print("✅ ICD-10 index built and cached to Drive")

## 5. Configure Training

In [ ]:
# ── Tune these as needed ───────────────────────────────────────────
EPOCHS        = 5          # 5 epochs with early stopping (patience=2)
BATCH_SIZE    = 16         # T4 has 16 GB VRAM — 16 fits comfortably
GRAD_ACCUM    = 1          # Effective batch = BATCH_SIZE * GRAD_ACCUM
LR            = 2e-5
MAX_LEN       = 512
STRIDE        = 256        # EDA: median span=2 words, 256 is safe

MODEL_OUT_DIR = 'models/ner/'
DRIVE_MODEL_DIR = '/content/drive/MyDrive/medical-coding-models/ner/'

# ── Set to an integer to use a sample (None = full dataset) ───────
MAX_TRAIN_SAMPLES = None   # e.g. 50 for a 3-minute smoke-test
MAX_DEV_SAMPLES   = None

os.makedirs('models/ner', exist_ok=True)
os.makedirs('outputs', exist_ok=True)
print("Config OK")

## 6. Train

In [ ]:
cmd = (
    f"python train_ner.py"
    f" --no_wandb"
    f" --epochs {EPOCHS}"
    f" --batch_size {BATCH_SIZE}"
    f" --grad_accum {GRAD_ACCUM}"
    f" --learning_rate {LR}"
    f" --max_length {MAX_LEN}"
    f" --stride {STRIDE}"
    f" --output_dir {MODEL_OUT_DIR}"
    f" --fp16"   # T4 supports fp16 — ~2x speedup
)
if MAX_TRAIN_SAMPLES:
    cmd += f" --max_train_samples {MAX_TRAIN_SAMPLES}"
if MAX_DEV_SAMPLES:
    cmd += f" --max_dev_samples {MAX_DEV_SAMPLES}"

print("Running:", cmd)
print("=" * 60)
!{cmd}

## 7. Save Model to Google Drive

In [ ]:
import shutil, json

# Save trained model to Drive so it persists after session ends
if os.path.exists(MODEL_OUT_DIR):
    os.makedirs(DRIVE_MODEL_DIR, exist_ok=True)
    shutil.copytree(MODEL_OUT_DIR, DRIVE_MODEL_DIR, dirs_exist_ok=True)
    print(f"✅ Model saved to Drive: {DRIVE_MODEL_DIR}")
else:
    print("⚠️  Model directory not found — did training complete?")

# Print final results
results_path = 'outputs/ner_results.json'
if os.path.exists(results_path):
    results = json.load(open(results_path))
    print("\n📊 Final Results")
    print(f"   Precision : {results.get('eval_precision', 0):.4f}")
    print(f"   Recall    : {results.get('eval_recall', 0):.4f}")
    print(f"   F1        : {results.get('eval_f1', 0):.4f}")
    print(f"   Loss      : {results.get('eval_loss', 0):.4f}")

## 8. (Optional) Download Model to Local Machine

In [ ]:
# Zip the model and download it locally
# Run this cell only if you want to pull the model back to your Mac

import shutil
from google.colab import files

model_zip = '/content/ner_model.zip'
shutil.make_archive('/content/ner_model', 'zip', MODEL_OUT_DIR)
print(f"Model zipped: {model_zip}")
files.download(model_zip)  # triggers browser download